# Обработка единичных пропусков в qty
Проблема: при обучении модели на выборке из 6 товаров возникла ситуация, что у товара с SKU = ЙО20155 (птичка из Йошкар-Олы) в 10.2025 не было продаж. Предположительно, они не поступили, а не то, что покупатели их не купили, так как товар очень ходовой. Такие скачки влияют на модель.

Решение:
- загружаем и агрегируем данные до sku-month
- строим полную месячную сетку
- ищем именно одиночные нули между двумя ненулевыми месяцами
- сохраняем их в отдельную переменную
- делаем удобную таблицу для ручной оценки
- разделяем на:
  - fill_by_interpolation
  - exclude_from_forecast

Гипотеза: если заполнить нулевые значения с помощью интерполяции по соседям, то модель будет выдавать более точный прогноз, так как факт отсутствия продаж является ложным

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

DATA_PATH = Path("../../data/clean/new_sales_data.csv")


In [2]:
def load_monthly_sales(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    df["sku"] = df["sku"].fillna("").astype(str).str.strip()
    df["product"] = df["product"].fillna("").astype(str).str.strip()
    df["unit"] = df["unit"].fillna("").astype(str).str.strip()
    df["month"] = pd.to_datetime(df["month"], format="%Y-%m")
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").fillna(0.0)

    df = df.loc[df["sku"] != ""].copy()

    monthly = (
        df.groupby(["sku", "month"], as_index=False)
        .agg(
            product=("product", "first"),
            unit=("unit", "first"),
            qty=("qty", "sum"),
        )
        .sort_values(["sku", "month"])
        .reset_index(drop=True)
    )
    return monthly


def expand_to_month_grid(monthly: pd.DataFrame) -> pd.DataFrame:
    all_months = pd.date_range(monthly["month"].min(), monthly["month"].max(), freq="MS")
    sku_meta = monthly[["sku", "product", "unit"]].drop_duplicates("sku")

    full_index = pd.MultiIndex.from_product(
        [sku_meta["sku"].tolist(), all_months],
        names=["sku", "month"],
    )

    expanded = (
        pd.DataFrame(index=full_index)
        .reset_index()
        .merge(sku_meta, on="sku", how="left")
        .merge(monthly[["sku", "month", "qty"]], on=["sku", "month"], how="left")
    )

    expanded["qty"] = expanded["qty"].fillna(0.0)
    expanded = expanded.sort_values(["sku", "month"]).reset_index(drop=True)
    return expanded


In [3]:
monthly = load_monthly_sales(DATA_PATH)
full_df = expand_to_month_grid(monthly)

print("monthly shape:", monthly.shape)
print("full_df shape:", full_df.shape)
print("sku count:", full_df["sku"].nunique())
print("months:", full_df["month"].min(), "->", full_df["month"].max())

full_df.head()


monthly shape: (16066, 5)
full_df shape: (34308, 5)
sku count: 953
months: 2023-01-01 00:00:00 -> 2025-12-01 00:00:00


,sku,month,product,unit,qty
0,0104,2023-01-01,"УксусБассо Balsamic 0,5 л",шт,5.0
1,0104,2023-02-01,"УксусБассо Balsamic 0,5 л",шт,2.0
2,0104,2023-03-01,"УксусБассо Balsamic 0,5 л",шт,3.0
3,0104,2023-04-01,"УксусБассо Balsamic 0,5 л",шт,1.0
4,0104,2023-05-01,"УксусБассо Balsamic 0,5 л",шт,0.0


In [4]:
def detect_single_zero_gaps(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(["sku", "month"]).reset_index(drop=True)

    grp = df.groupby("sku", sort=False)["qty"]

    df["prev_qty"] = grp.shift(1)
    df["next_qty"] = grp.shift(-1)
    df["prev_month"] = df.groupby("sku", sort=False)["month"].shift(1)
    df["next_month"] = df.groupby("sku", sort=False)["month"].shift(-1)

    # Одиночный ноль между двумя ненулевыми соседями
    df["single_zero_gap"] = (
        (df["qty"] == 0) &
        (df["prev_qty"] > 0) &
        (df["next_qty"] > 0)
    )

    return df


In [7]:
gap_df = detect_single_zero_gaps(full_df)

single_zero_gaps = (
    gap_df.loc[gap_df["single_zero_gap"]].copy()
    [["sku", "product", "unit", "month", "qty", "prev_qty", "next_qty"]]
    .sort_values(["sku", "month"])
    .reset_index(drop=True)
)

print("Количество одиночных нулевых месяцев:", len(single_zero_gaps))
single_zero_gaps

Количество одиночных нулевых месяцев: 699


,sku,product,unit,month,qty,prev_qty,next_qty
0,0104,"УксусБассо Balsamic 0,5 л",шт,2023-09-01,0.0,3.0,1.0
1,0104,"УксусБассо Balsamic 0,5 л",шт,2024-02-01,0.0,2.0,1.0
2,15033,Какао-напиток растворимый БЕЛЫЙ МИШКА 300 г,шт,2025-11-01,0.0,3.0,5.0
3,38010,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-04-01,0.0,5.0,4.0
4,38015,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-10-01,0.0,12.0,22.0
...,...,...,...,...,...,...,...
694,ЯП25162,Печенье АЛЕНКА с начинкой со вкусом соленой ка...,шт,2025-06-01,0.0,3.0,6.0
695,ЯП25162,Печенье АЛЕНКА с начинкой со вкусом соленой ка...,шт,2025-11-01,0.0,21.0,7.0
696,ЯП25415,Печенье Аленка Брауни сдобное мягкое с шоколад...,шт,2025-07-01,0.0,12.0,11.0
697,ЯП25623,Пряник ТУЛЬСКИЙ с фруктовой начинкой 130 гр,шт,2025-09-01,0.0,1.0,24.0


In [8]:
single_zero_gaps = single_zero_gaps.copy()

single_zero_gaps["neighbor_mean"] = (
    single_zero_gaps["prev_qty"] + single_zero_gaps["next_qty"]
) / 2

single_zero_gaps["neighbor_abs_diff"] = (
    single_zero_gaps["prev_qty"] - single_zero_gaps["next_qty"]
).abs()

single_zero_gaps["neighbor_rel_diff_%"] = np.where(
    single_zero_gaps["neighbor_mean"] > 0,
    single_zero_gaps["neighbor_abs_diff"] / single_zero_gaps["neighbor_mean"] * 100,
    np.nan,
)

single_zero_gaps["suggest_fill"] = single_zero_gaps["neighbor_rel_diff_%"] <= 40

single_zero_gaps.head(30)


,sku,product,unit,month,qty,prev_qty,next_qty,neighbor_mean,neighbor_abs_diff,neighbor_rel_diff_%,suggest_fill
0,0104,"УксусБассо Balsamic 0,5 л",шт,2023-09-01,0.0,3.000000,1.000,2.000000,2.000000,100.000000,False
1,0104,"УксусБассо Balsamic 0,5 л",шт,2024-02-01,0.0,2.000000,1.000,1.500000,1.000000,66.666667,False
2,15033,Какао-напиток растворимый БЕЛЫЙ МИШКА 300 г,шт,2025-11-01,0.0,3.000000,5.000,4.000000,2.000000,50.000000,False
3,38010,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-04-01,0.0,5.000000,4.000,4.500000,1.000000,22.222222,True
4,38015,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-10-01,0.0,12.000000,22.000,17.000000,10.000000,58.823529,False
5,38030,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2025-04-01,0.0,6.000000,3.000,4.500000,3.000000,66.666667,False
6,9010,"имбирь томленый, пр-во Россия",шт,2024-12-01,0.0,9.000000,12.000,10.500000,3.000000,28.571429,True
7,9010,"имбирь томленый, пр-во Россия",шт,2025-09-01,0.0,11.000000,25.000,18.000000,14.000000,77.777778,False
8,9020,Мармелад в стеклянной банке 250 мл,шт,2024-08-01,0.0,55.000000,43.000,49.000000,12.000000,24.489796,True
9,9127275,Цикорий гранулированный А.П. СЕЛИВАНОВ Императ...,шт,2024-10-01,0.0,20.000000,20.000,20.000000,0.000000,0.000000,True


In [9]:
# Сводка по SKU: у кого сколько таких подозрительных месяцев
gap_summary = (
    single_zero_gaps.groupby(["sku", "product", "unit"], as_index=False)
    .agg(
        single_zero_gap_count=("month", "count"),
        first_gap_month=("month", "min"),
        last_gap_month=("month", "max"),
    )
    .sort_values(["single_zero_gap_count", "sku"], ascending=[False, True])
    .reset_index(drop=True)
)

gap_summary.head(30)


,sku,product,unit,single_zero_gap_count,first_gap_month,last_gap_month
0,РФ20960,Халва Абрикосовские сладости воздушная с арахи...,шт,7,2023-05-01,2025-11-01
1,ББ20058,Шоколад ВДОХНОВЕНИЕ темный классический 60 г,шт,6,2023-06-01,2025-09-01
2,КО22001,КОНФ Батончики со сниженным сахаром Украли сах...,шт,6,2023-08-01,2025-11-01
3,ББ15827,КОНФ КОР ВДОХНОВЕНИЕ 400 гр,шт,5,2024-02-01,2025-10-01
4,ВО00560,КОНФ КОР Птичье молоко 300 гр РОТ ФРОНТ,шт,5,2024-06-01,2025-10-01
5,КО04780,КОНФ КОР Птичье молоко 300 гр,шт,5,2023-09-01,2025-06-01
6,КО05702,КОНФ КОР Тарханы 450 гр,шт,5,2023-02-01,2025-07-01
7,КО11301,КОНФ КОР Старинная открытка пенал 75г,шт,5,2023-07-01,2025-04-01
8,КО17011,КОНФ КОР Набор конф Москва Счастливые моменты ...,шт,5,2023-02-01,2025-06-01
9,НС20883,Шоколад Сибирский сувенир 340 гр,шт,5,2023-06-01,2025-07-01


In [10]:
# Отдельная переменная для ручной оценки
# Эту таблицу удобно просматривать и потом руками проставлять решение
manual_review_df = single_zero_gaps.copy()

manual_review_df["decision"] = ""
manual_review_df["comment"] = ""

manual_review_df.head(30)


,sku,product,unit,month,qty,prev_qty,next_qty,neighbor_mean,neighbor_abs_diff,neighbor_rel_diff_%,suggest_fill,decision,comment
0,0104,"УксусБассо Balsamic 0,5 л",шт,2023-09-01,0.0,3.000000,1.000,2.000000,2.000000,100.000000,False,,
1,0104,"УксусБассо Balsamic 0,5 л",шт,2024-02-01,0.0,2.000000,1.000,1.500000,1.000000,66.666667,False,,
2,15033,Какао-напиток растворимый БЕЛЫЙ МИШКА 300 г,шт,2025-11-01,0.0,3.000000,5.000,4.000000,2.000000,50.000000,False,,
3,38010,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-04-01,0.0,5.000000,4.000,4.500000,1.000000,22.222222,True,,
4,38015,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2024-10-01,0.0,12.000000,22.000,17.000000,10.000000,58.823529,False,,
5,38030,Смесь для коктейля гранулированная БЕЛЫЙ МИШКА...,шт,2025-04-01,0.0,6.000000,3.000,4.500000,3.000000,66.666667,False,,
6,9010,"имбирь томленый, пр-во Россия",шт,2024-12-01,0.0,9.000000,12.000,10.500000,3.000000,28.571429,True,,
7,9010,"имбирь томленый, пр-во Россия",шт,2025-09-01,0.0,11.000000,25.000,18.000000,14.000000,77.777778,False,,
8,9020,Мармелад в стеклянной банке 250 мл,шт,2024-08-01,0.0,55.000000,43.000,49.000000,12.000000,24.489796,True,,
9,9127275,Цикорий гранулированный А.П. СЕЛИВАНОВ Императ...,шт,2024-10-01,0.0,20.000000,20.000,20.000000,0.000000,0.000000,True,,


In [11]:
# Если хотите посмотреть конкретный SKU в истории целиком
sku_to_check = "ЙО20155"

gap_df.loc[gap_df["sku"] == sku_to_check, ["sku", "product", "month", "qty"]].reset_index(drop=True)


,sku,product,month,qty
0,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-01-01,12.949000
1,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-02-01,16.001000
2,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-03-01,21.804000
3,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-04-01,17.746000
4,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-05-01,26.516000
5,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-06-01,16.901000
6,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-07-01,23.379000
7,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-08-01,8.063000
8,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-09-01,0.000000
9,ЙО20155,КОНФ ВЕС Птичье молоко по ГОСТ,2023-10-01,8.211000


In [12]:
# Пример: сюда потом руками вносите решения
# decision:
# - "fill"    -> можно заполнять интерполяцией
# - "exclude" -> SKU не прогнозируем
# - "keep"    -> оставляем как есть

manual_decisions = pd.DataFrame(
    [
        # ("ЙО20155", "2025-10-01", "fill", "нет поставки, можно интерполировать"),
        # ("ABC123", "2024-06-01", "exclude", "слишком нестабильный SKU"),
    ],
    columns=["sku", "month", "decision", "comment"]
)

if not manual_decisions.empty:
    manual_decisions["month"] = pd.to_datetime(manual_decisions["month"])

manual_decisions


,sku,month,decision,comment


## SKU с риском шумных нулей для модели

Эта секция помогает быстро найти товары, где одиночные нули особенно вероятно искажают лаги, rolling mean и итоговый прогноз. Такие SKU лучше:
- вручную проверить,
- не обучать на глубоких деревьях без сглаживания,
- дополнительно фильтровать через sparse-rule.


In [ ]:
sku_history_months = full_df.groupby("sku").size().rename("history_months")

gap_risk = (
    single_zero_gaps.groupby(["sku", "product", "unit"], as_index=False)
    .agg(
        single_zero_gap_count=("month", "count"),
        avg_neighbor_mean=("neighbor_mean", "mean"),
        avg_neighbor_abs_diff=("neighbor_abs_diff", "mean"),
    )
    .merge(sku_history_months, on="sku", how="left")
)

gap_risk["single_zero_gap_share"] = gap_risk["single_zero_gap_count"] / gap_risk["history_months"]
gap_risk["risk_label"] = np.select(
    [
        (gap_risk["single_zero_gap_count"] >= 4) | (gap_risk["single_zero_gap_share"] >= 0.12),
        (gap_risk["single_zero_gap_count"] >= 2) | (gap_risk["single_zero_gap_share"] >= 0.06),
    ],
    ["high", "medium"],
    default="low",
)

gap_risk.sort_values(
    ["risk_label", "single_zero_gap_count", "single_zero_gap_share"],
    ascending=[True, False, False],
).head(30)


In [ ]:
gap_risk_summary = (
    gap_risk.groupby("risk_label", observed=False)
    .agg(
        sku_count=("sku", "count"),
        mean_gap_count=("single_zero_gap_count", "mean"),
        mean_gap_share=("single_zero_gap_share", "mean"),
        mean_neighbor_mean=("avg_neighbor_mean", "mean"),
    )
    .reset_index()
)

gap_risk_summary
